# FastSpeech 2 - English (Svarah) + Telugu (IndicTTS)
Runtime -> Change runtime type -> **GPU (T4)**. Run top to bottom. Set `REPO` to your GitHub repo URL (push this project there first) or upload the folder to Drive.

In [ ]:
REPO='https://github.com/<your-user>/<your-repo>.git'
!git clone $REPO /content/project
%cd /content/project
!pip -q install -r requirements.txt
!apt-get -qq install -y fonts-noto-core >/dev/null  # Telugu font for plots
import nltk
for p in ['averaged_perceptron_tagger_eng','cmudict','punkt']: nltk.download(p)

## 1. Hugging Face login (Svarah is gated: accept the terms on its dataset page first)

In [ ]:
from huggingface_hub import login
login()

## 2. Montreal Forced Aligner (conda-forge). condacolab restarts the kernel once: after it restarts, re-run from the `%cd` cell and skip this install cell.

In [ ]:
!pip -q install condacolab
import condacolab; condacolab.install()

In [ ]:
%cd /content/project
!mamba install -q -y -c conda-forge montreal-forced-aligner
!mfa version

## 3. Preprocess (both languages)

In [ ]:
%cd /content/project/English
!python preprocess.py all -j 4
%cd /content/project/Telugu
!python preprocess.py all -j 4

## 4. Train (about 1-2 h each on a T4). Checkpoints/logs land in `checkpoints/` and `logs/`. Re-run with `--restore_step N` if Colab disconnects.

In [ ]:
%cd /content/project/English
!python train.py

In [ ]:
%cd /content/project/Telugu
!python train.py

## 5. Inference: audio + duration/pitch/energy plots into `Results/<Language>/`

In [ ]:
%cd /content/project/English
!python inference.py
%cd /content/project/Telugu
!python inference.py

## 6. Download everything (or copy to Drive)

In [ ]:
%cd /content/project
!zip -qr results_and_ckpts.zip Results English/checkpoints Telugu/checkpoints English/logs Telugu/logs English/data/preprocessed/*.json Telugu/data/preprocessed/*.json
from google.colab import files; files.download('results_and_ckpts.zip')